# Elaborato Autonomous Multi Agent

Questo notebook contiene il progetto per il corso di Autonoumus Multi Agent. 
Per questo elaborato si è simulare e analizzato l'applicazione dell'algoritmo di RL **placeholder** per andare ad automatizzare in modo "intelligente" il comportamento di un ascensore; il lavoro è diviso in tre parti principali:

1. La definizione del comportamento dell'ascensore e del relativo ambiente virtuale.

2. Un richiamo teorico su **placeholder**.

3. Un'analisi approfondita sull'addestramento e il tuning degli iperparametri del modello.

Infine, è inoltre presente una piccola sezione di conclusioni che riassume i fenomeni salienti osservati durante lo sviluppo del progetto.

## Albero Cartelle

```text
ascensori-rl/
├── Elevator.ipynb
├── .gitignore
└── README.md
```

## Risorse Utilizzate

- https://gymnasium.farama.org/

## Dipendenze da installare

In [81]:
%pip install numpy gymnasium

Note: you may need to restart the kernel to use updated packages.


# Prima Parte: Alti e Bassi

In questa sezione del notebook andiamo a definire il comportamento dell'ascensore e dell'ambiente che questo andrà a servire: un palazzo di 5 piani e i suoi, più o meno numerosi, passeggeri.
Per definire tale ambiente è stata disposta una coda per ciascun piano del palazzo che modella la coda di passeggeri in attesa davanti all'ascensore, i passeggeri vengono generati in modo casuale tramite una distribuzione di Poisson, la quale, durante ogni intervallo temporale, determina il numero di nuovi arrivi a ciascun piano. La destinazione di ciascun passeggero viene invece estratta quando quest'ultimo sale sull'ascensore, utilizzando una matrice di probabilità origine-destinazione $P_{OD}$.

Lo stato osservabile dell'ambiente sarà dunque composto come segue:

$$
S = \{coda_0, coda_1, coda_2, coda_3, coda_4, attesa_0, attesa_1, attesa_2, attesa_3, attesa_4, piano\ attuale, piano\ destinazione, passeggero\ a\ bordo\}
$$

Per ogni piano l'agente osserva dunque sia **quanti** passeggeri sono in coda sia **da quanto tempo** aspetta il passeggero che è arrivato per primo: $attesa_i$ è infatti il tempo di attesa, in secondi, del passeggero che aspetta da più tempo al piano $i$ (se la coda è vuota, è pari a 0). Il campo relativo alla destinazione rappresenta la destinazione del passeggero a bordo, quando l'ascensore è vuoto questo coincide con il piano attuale dell'ascensore. L'ascensore può inoltre trasportare **un solo passeggero** alla volta, per questo il campo relativo al passeggero a bordo può assumere solo i valori 0 e 1.

L'ascensore si sposta di un piano per ogni azione di salita o discesa e ciascuno step della simulazione corrisponde a un intervallo temporale di 5 secondi. L'imbarco avviene quando l'ascensore è vuoto **e** si trova a un piano con almeno un passeggero in attesa **e** |l'agente sceglie di restare fermo **o** ha appena sbarcato un altro passeggero|, lo sbarco, invece, avviene automaticamente quando l'ascensore raggiunge il piano di destinazione; in questo modo si è cercato di modellare sia la possibilità di transitare da un piano con passeggeri in attesa **senza necessariamente fermarsi** sia il fatto che, una volta portato a destinazione un passeggero in un piano con altri passeggeri in attesa, lo scambio tra chi esce e chi entra avvenisse in un **solo step**.

Ciò che vogliamo rendere efficiente è la gestione delle code ai vari piani, in particolare vorremo che le code non fossero troppo lunghe ma neanche che vi fossero persone che aspettano da troppo tempo; se si usasse come reward $r =-N$ avremmo code meno numerose ma, potrebbe capitare di avere passeggeri che aspettano molto tempo in attesa, se, al contrario, si usasse $R_t = -\sum_{i \in Q_t}(t-t_i^{arrivo})$ (dove $Q_t$ rappresenta l'insieme dei passeggeri ancora in attesa al termine dello step), si penalizzerebbero maggiormente le attese lunghe, questo però potrebbe portare a trascurare maggiormente le code più lunghe, le quali sul lungo periodo potrebbero causare un rallentamento.

Per questo motivo, si è cercato di definire una reward che considerasse entrambe queste esigenze:

$$
R_t = -\left(N_t + \lambda \frac{W_t}{T_{rif}}\right)
$$

dove $W_t$ rappresenta la somma dei tempi di attesa di tutti i passeggeri ancora in coda, $T_{rif}$ è un tempo di riferimento espresso in secondi e $\lambda$ è un parametro che regola il peso attribuito alle attese prolungate.

In questo modo, la reward penalizza sia la presenza di numerosi passeggeri in attesa sia il protrarsi delle attese nel tempo. Il parametro $\lambda$ permette di regolare il contributo del secondo termine, mentre la normalizzazione tramite $T_{rif}$ rende le due componenti più facilmente confrontabili.

Infine, le possibili azioni dell'ascensore sono le seguenti:

$$
A = \{Resta\ fermo,\ Sale,\ Scende\}
$$

In [69]:
#import delle librerie necessarie
import sys
from typing import Optional
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt


#parametri fondamentali dell'ambiente:

#numero dei piani del palazzo
n_floors = 5
#durata dell'episodio(steps dell'episodio)
max_steps = 500

#matrice congiunta origine-destinazione
#P_OD[i, j] = probabilità congiunta di origine i e destinazione j
#le righe rappresentano i piani di partenza,
#mentre le colonne rappresentano i piani di destinazione
#
#ipotesi iniziali:
# - i cinque piani hanno la stessa probabilità di origine
# - dai piani più bassi si viaggi verso i piani più alti
#   con p crescente ("per un piano prendo le scale")
# - dai piani più alti si viaggi verso i piani più bassi
#   con p crescente ("per un piano prendo le scale")
#(- il palazzo possiede anche delle scale)
# - dal piano intermedio si viaggia con p uguali
#   e verso l'alto e verso il basso

P_OD = np.array([
    
    #a PT  a P1  a P2  a P3  a P4
    [0.00, 0.02, 0.04, 0.06, 0.08], #da PT
    [0.02, 0.00, 0.03, 0.06, 0.09], #da P1
    [0.05, 0.05, 0.00, 0.05, 0.05], #da P2
    [0.08, 0.06, 0.04, 0.00, 0.02], #da P3
    [0.08, 0.06, 0.04, 0.02, 0.00]  #da P4
    
], dtype=np.float32)


#controlli sui valori presenti nella matrice
if np.any(P_OD < 0):
    raise ValueError("Le probabilità non possono essere negative.")

if not np.allclose(np.diag(P_OD), 0.0):
    raise ValueError("La diagonale deve essere nulla.")

if not np.isclose(P_OD.sum(), 1.0):
    raise ValueError("La somma totale di P_OD deve essere 1.")

#si definisce poi il tasso medio di arrivo dei passeggeri per piano
#(lambda = numero medio di arrivi per intervallo temporale)
#questo ci servirà dopo per modellare l'arrivo dei passeggeri
#per i quali verrà impiegata una distribuzione di Poisson

arrival_rate_per_minute = np.full(
    N_FLOORS, 0.5, dtype=np.float32
)

#controlli e stampe

assert np.all(np.isfinite(arrival_rate_per_minute))
assert np.all(arrival_rate_per_minute >= 0)

print("Matrice origine-destinazione P_OD:")
print(P_OD)

print("\nSomma totale di P_OD:")
print(P_OD.sum())


Matrice origine-destinazione P_OD:
[[0.   0.02 0.04 0.06 0.08]
 [0.02 0.   0.03 0.06 0.09]
 [0.05 0.05 0.   0.05 0.05]
 [0.08 0.06 0.04 0.   0.02]
 [0.08 0.06 0.04 0.02 0.  ]]

Somma totale di P_OD:
1.0


## Gymansium: Costruendo la Palestra

Per modellare l'ambiente è stata utilizzata la libreria Python **Gymnasium**, che mette a disposizione un'interfaccia standard per la definizione di ambienti per il *Reinforcement Learning* (RL), in questo modo è possibile separare la logica dell'ambiente da quella dell'agente e utilizzare algoritmi di apprendimento compatibili con la libreria.

Per costruire l'ambiente è stata definita la classe ElevatorEnv, ereditando gym.Env, questo ci consente di avere accesso alle funzionalità di base previste da Gymnasium e viene garantita la compatibilità con gli strumenti che implementano algoritmi di RL.

All'interno della classe sono stati implementati i seguenti metodi principali:

- __init\__(): viene inizializzato l'ambiente, definendo le caratteristiche dell'edificio, le code dei passeggeri, i parametri del processo di arrivo e gli spazi delle azioni e delle osservazioni.
- reset(): viene inizializzata una nuova simulazione, ripristinando le condizioni iniziali dell'ambiente e azzerando le informazioni accumulate durante l'episodio precedente. Vengono inoltre restituite la prima osservazione e un dizionario contenente informazioni aggiuntive.
- step(action): viene eseguito un passo della simulazione sulla base dell'azione selezionata dall'agente. In questo metodo vengono gestiti lo spostamento dell'ascensore, l'imbarco e lo sbarco dei passeggeri, l'arrivo di nuovi passeggeri e il calcolo della reward.

Al termine di ogni chiamata al metodo step(), vengono restituiti cinque elementi, secondo l'interfaccia prevista da Gymnasium: observation, reward, terminated, truncated e info. In particolare, terminated indica il raggiungimento di una condizione terminale dell'ambiente, mentre truncated segnala l'interruzione dell'episodio dovuta al raggiungimento del numero massimo di step previsto; in questo caso verrà utilizzato truncated in quanto si è immaginato che il flusso dei passeggeri fosse continuo.

Una volta definiti i metodi principali e gli spazi delle azioni e delle osservazioni, l'ambiente può essere istanziato e utilizzato per l'addestramento di un agente. Durante l'apprendimento, l'agente riceve le osservazioni dell'ambiente, seleziona le azioni da eseguire e ottiene una reward in funzione delle conseguenze delle proprie decisioni.

In [70]:
#costruiamo la classe che rappresenta
#l'ambiente dell'ascensore

class ElevatorEnv(gym.Env):

    def __init__(
        self,
        p_od,
        n_floors=5,
        arrival_rate_per_minute=0.5,
        step_duration_seconds=5,
        max_steps=500,
        age_weight=0.5,
        reference_wait_seconds=60.0
    ):
        
        #definizione valori degli episodi
        self.step_duration_seconds = float(step_duration_seconds)
        self.max_steps = int(max_steps)
        
        # si definiscono i parametri della reward
        self.age_weight = float(age_weight)
        self.reference_wait_seconds = float(reference_wait_seconds)
        
        #si definisce ciò che riguarda
        #il palazzo e l'ascensore,
        #in seguito ci si occuperà dei passeggeri
    
        #numero dei piani del palazzo
        self.n_floors = int(n_floors)
    
        #si tiene traccia del piano a cui si trova,
        #di quello a cui deve dirigersi,
        #e della presenza o meno del passeggero
        self._agent_location = 0
        self._target_location = 0
        self._is_full = False
    
        #si definisce poi una coda per ciascun piano:
        #ogni coda è la lista degli step di arrivo
        #dei passeggeri in attesa, in questo modo la lunghezza
        #della lista è il numero di passeggeri in coda
        #e il primo elemento è il passeggero che aspetta
        #da più tempo; gli step di arrivo servono
        #anche per definire la reward
        self.queues = [[] for _ in range(self.n_floors)]
            
        #si definisce ciò che l'agente può osservare:
        #per ogni piano il numero di passeggeri in coda (q)
        #e il tempo di attesa, in secondi, del passeggero
        #che aspetta da più tempo (w), che non può superare
        #la durata dell'episodio
        self.observation_space = gym.spaces.Dict(
            {
                "agent" : gym.spaces.Discrete(self.n_floors),
                "target": gym.spaces.Discrete(self.n_floors),
                "passenger" : gym.spaces.Discrete(2),
                **{
                    f"q{i+1}" : gym.spaces.Box(
                        low=0,
                        shape=(),
                        high=np.iinfo(np.int64).max,
                        dtype=np.int64
                    )
                    for i in range(self.n_floors) 
                },
                **{
                    f"w{i+1}" : gym.spaces.Box(
                        low=0.0,
                        shape=(),
                        high=self.max_steps * self.step_duration_seconds,
                        dtype=np.float32
                    )
                    for i in range(self.n_floors) 
                }
            }
        )
    
        #si definiscono le azioni possibili:
        # 0 - resta fermo
        # 1 - sale
        # 2 - scende
        self.action_space = gym.spaces.Discrete(3)
    
        self._action_to_direction = {
            0: 0,
            1: 1,
            2: -1
        }
    
        #vengono poi trattati i passeggeri
    
        #qui abbiamo la matrice di origine-destinazione,
        #che servirà in seguito per campionare i passeggeri
        p_od = np.asarray(p_od, dtype=np.float64)
        row_sums = p_od.sum(axis=1)

        #le righe vengono normalizzate,
        #così che P_OD[i] sia direttamente la distribuzione
        #delle destinazioni per un passeggero che parte dal piano i
        self.P_OD = p_od / row_sums[:, None]
            
        #tasso di arrivo medio al minuto
        #(ex.: 0,5 = un passeggero ogni due minuti)
        self.arrival_rate_per_minute = np.full(
            self.n_floors,
            float(arrival_rate_per_minute),
            dtype=np.float32
        )
    
        #gli arrivi medi al minuto sono convertiti
        #in arrivi medi per step
        self.arrival_rate = (
            self.arrival_rate_per_minute
            * self.step_duration_seconds / 60.0
        )


    #si definisce il metodo che ritorna l'osservazione
    def _get_obs(self):

        #l'attesa del passeggero più vecchio si ricava dal primo
        #elemento della coda, se la coda è vuota vale 0
        #(la coda vuota si distingue comunque grazie al conteggio q)
        return {
            "agent" : np.int64(self._agent_location),
            "target": np.int64(self._target_location),
            "passenger": np.int64(self._is_full),
            **{
                f"q{i+1}" : np.int64(len(queue))
                for i, queue in enumerate(self.queues)
            },
            **{
                f"w{i+1}" : np.float32(
                    (self.current_step - queue[0])
                    * self.step_duration_seconds
                    if queue else 0.0
                )
                for i, queue in enumerate(self.queues)
            }
        }

    #si definisce l'inizializzazione di un nuovo episodio
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):

        super().reset(seed=seed)

        #si inizializza casualmente il piano di partenza
        #che all'inizio coinciderà col piano target
        #in quanto l'ascensore sarà inizialmente vuoto
        self._is_full = False
        self._agent_location = int(self.np_random.integers(0, self.n_floors))
        self._target_location = self._agent_location

        #le code dei piani vengono re-inizializzate
        # DA MODIFICARE [se funziona sarebbe interessante
        # testare con code già formate]
        self.queues = [[] for _ in range(self.n_floors)]

        self.waiting_times = []

        #si inizializza il contatore degli step
        self.current_step = 0

        observation = self._get_obs()
        info = {}

        return observation, info

    #si definisce ora lo step dell'espisodio
    def step(self, action):

        #si controlla che l'azione scelta appartenga
        #allo spazio delle azioni possibili
        if not self.action_space.contains(action):
            raise ValueError(f"Azione non valida: {action}")

        action = int(action)
        last_waiting_time = None
        
        #viene mappata l'azione discreta alla direzione del movimento
        direction = self._action_to_direction[action]

        #viene incrementato il contatore degli step,
        #da qui in poi current_step è l'istante
        #a cui avviene la transizione
        self.current_step += 1

        #si aggiorna la posizione dell'ascensore,
        #controllando i limiti dello spazio:
        self._agent_location = int(np.clip(
            self._agent_location + direction, 0, self.n_floors - 1
        ))

        #flag che mi serve per fare in modo che,
        #se un passeggero è appena sbarcato,
        #un altro passeggero può essere
        #imbarcato immediatamente

        #se l'ascensore è pieno ed è arrivato a destinazione
        #il passeggero scende
        passenger_unloaded = (
            self._is_full
            and self._agent_location == self._target_location
        )

        if passenger_unloaded:
            self._is_full = False

        #se:
        #- l'ascensore è vuoto
        #- vi è almeno un passeggero in coda al piano corrente
        #- l'ascensore ha deciso di rimanere fermo o è appena sbarcato un passeggero
        #il passeggero viene imbarcato e viene impostato il piano d'arrivo
        floor = self._agent_location

        if (
            not self._is_full 
            and self.queues[floor]
            and (action == 0 or passenger_unloaded)
            ):

            #viene recuperato il tempo d'arrivo del passeggero
            #e contemporaneamente rimosso dalla coda
            arrival_step = self.queues[floor].pop(0)

            #si calcola poi il tempo di attesa totale
            #dell'ultimo passeggero imbarcato
            last_waiting_time = (
                self.current_step - arrival_step
            ) * self.step_duration_seconds

            self.waiting_times.append(last_waiting_time)

            #viene poi campionata la destinazione del passeggero
            #appena imbarcato
            self._target_location = int(
                self.np_random.choice(
                    self.n_floors,
                    p=self.P_OD[floor]
                )
            )

            #l'ascensore è occupato
            self._is_full = True

        #se l'ascensore non è pieno, il suo target equivale alla sua posizione
        if not self._is_full:
            self._target_location = self._agent_location

        #si simula poi il possibile arrivo di nuovi passeggeri,
        #per definire l'arrivo dei passeggeri si utilizza
        #una distribuzione di Poisson
        new_passengers = self.np_random.poisson(
            lam=self.arrival_rate
        )

        #vengono poi enumerate le quantità di passeggeri che arrivano 
        #a ciascun piano e, per ciascun passeggero,
        #viene aggiunto alla coda del piano a cui arriva
        #lo step al quale arriva
        for floor, n_arrivals in enumerate(new_passengers):
            self.queues[floor].extend(
                [self.current_step] * int(n_arrivals)
            )

        #qui viene calcolata la reward sullo stato risultante dalla transizione

        #si calcola il numero totale di passeggeri in attesa
        total_waiting_passengers = sum(
            len(queue) for queue in self.queues
        )
        
        #si calcola la somma dei tempi di attesa di tutti
        #i passeggeri ancora in coda espressa in secondi
        total_waiting_time = sum(
            self.current_step - arrival_step
            for queue in self.queues
            for arrival_step in queue
        ) * self.step_duration_seconds
        
        #si combinano la numerosità delle code e i tempi di attesa
        reward = -(
            total_waiting_passengers
            + self.age_weight
            * total_waiting_time
            / self.reference_wait_seconds
        )

        #non utilizziamo una terminazione naturale
        terminated = False
        #ma utilizziamo un troncamento della simulazione
        truncated = self.current_step >= self.max_steps

        #costruiamo una nuova osservazione
        observation = self._get_obs()

        info = {
            "current_step": self.current_step,
            "queues_total": sum(len(queue) for queue in self.queues),
            "passenger_onboard": self._is_full,
            "waiting_time_last_boarded": last_waiting_time,
            "passengers_boarded": len(self.waiting_times),
            "mean_waiting_time": (
                float(np.mean(self.waiting_times))
                if self.waiting_times else 0.0
            ),
            "total_waiting_age": total_waiting_time
        }

        return observation, reward, terminated, truncated, info

In [71]:
#si registra l'ambiente in gymnasium, così da poterlo
#creare con gym.make() tramite il suo identificativo
env_id = "Elevator-v0"

gym.register(
    id=env_id,
    entry_point=ElevatorEnv
)

### Testing dell'ambiente

Vengono adesso svolti i seguenti test sull'ambiente definito sinora al fine di verificarne la correttezza:

1. test di correttezza sintattica con *check_env()* da gymnasium
2. verifica del funzionamento delle azioni
3. verifica dell'arrivo dei passeggeri
4. verifica dell'imbarco
5. verifica dello sbarco
6. verifica della reward
7. verifica dell'osservazione
8. test di coerenza dello stato


#### Test con check_env()

Viene testato se l'ambiente è sintatticamente corretto.

In [72]:
#viene importato da gymnasium il metodo per verificare
#che l'ambiente sia sintatticamente corretto
from gymnasium.utils.env_checker import check_env

#viene creato un ambiente da testare
env_debug = ElevatorEnv(p_od=P_OD)

#si verifica che l'ambiente rispetti l'API di Gymnasium
try:
    check_env(env_debug, skip_render_check=True)
    print("Environment passes all checks!")

#se c'è un errore
except Exception as e:
    #mostriamo il tipo di errore e il messaggio
    print(f"Environment has issues: {type(e).__name__}: {e}")

Environment passes all checks!


#### Test delle azioni

Si verifica che le azioni abbiano l'effetto desiderato sugli spostamenti dell'ascensore e che, soprattutto, non si oltrepassino i limiti dell'edificio.

In [73]:
#si crea l'ambiente utilizzando il suo identificativo
env = gym.make(
    env_id,
    p_od=P_OD,
    max_steps=500
)

#si inizializza un episodio con un seed riproducibile
obs, info = env.reset(seed=42)

print("Osservazione iniziale:", obs)
print(
    "Osservazione valida:",
    env.observation_space.contains(obs)
)

#si controlla che l'osservazione iniziale sia valida
assert env.observation_space.contains(obs)

#si definisce una sequenza di azioni da provare
#0 = resta fermo, 1 = sale, 2 = scende
actions = [0, 1, 2, 0, 2, 1, 1, 1, 1, 1]

#si esegue il test per ciascuna azione
for action in actions:

    #si salva il piano dell'ascensore prima dell'azione
    old_floor = obs["agent"]

    #si esegue uno step della simulazione
    obs, reward, terminated, truncated, info = env.step(action)

    #si raccolgono le code dei cinque piani
    queues = [
        int(obs[f"q{i+1}"])
        for i in range(env.unwrapped.n_floors)
    ]

    #si stampano i risultati dello step
    print(
        f"Step: {info['current_step']} | "
        f"Azione: {action} | "
        f"Piano: {old_floor} -> {obs['agent']} | "
        f"Target: {obs['target']} | "
        f"Passeggero: {obs['passenger']} | "
        f"Code: {queues} | "
        f"Reward: {reward:.2f}"
    )

    #si verifica che la nuova osservazione sia valida
    assert env.observation_space.contains(obs)

    if terminated or truncated:
        print("Episodio terminato.")
        break

env.close()

Osservazione iniziale: {'agent': np.int64(0), 'target': np.int64(0), 'passenger': np.int64(0), 'q1': np.int64(0), 'q2': np.int64(0), 'q3': np.int64(0), 'q4': np.int64(0), 'q5': np.int64(0), 'w1': np.float32(0.0), 'w2': np.float32(0.0), 'w3': np.float32(0.0), 'w4': np.float32(0.0), 'w5': np.float32(0.0)}
Osservazione valida: True
Step: 1 | Azione: 0 | Piano: 0 -> 0 | Target: 0 | Passeggero: 0 | Code: [0, 0, 0, 0, 1] | Reward: -1.00
Step: 2 | Azione: 1 | Piano: 0 -> 1 | Target: 1 | Passeggero: 0 | Code: [0, 0, 0, 0, 1] | Reward: -1.04
Step: 3 | Azione: 2 | Piano: 1 -> 0 | Target: 0 | Passeggero: 0 | Code: [0, 0, 0, 0, 1] | Reward: -1.08
Step: 4 | Azione: 0 | Piano: 0 -> 0 | Target: 0 | Passeggero: 0 | Code: [0, 0, 0, 0, 1] | Reward: -1.12
Step: 5 | Azione: 2 | Piano: 0 -> 0 | Target: 0 | Passeggero: 0 | Code: [1, 0, 0, 0, 1] | Reward: -2.17
Step: 6 | Azione: 1 | Piano: 0 -> 1 | Target: 1 | Passeggero: 0 | Code: [1, 0, 0, 1, 1] | Reward: -3.25
Step: 7 | Azione: 1 | Piano: 1 -> 2 | Target:

#### Test sull'arrivo dei passeggeri

Viene verficata che la generazione dei passeggeri avvenga correttamente, in particolare si verifca che:

- il numero di passeggeri in coda riportato nell'osservazione corrisponda al numero di istanti di arrivo registrati in ogni coda
- tutti i nuovi arrivi siano stati registrati alla fine del primo step
- l'attesa osservata sia nulla, dato che i passeggeri sono appena arrivati

In [74]:
#si crea un ambiente con un tasso di arrivo elevato
#per verificare il funzionamento della distribuzione di Poisson
env_arrivi = ElevatorEnv(
    p_od=P_OD,
    arrival_rate_per_minute=60.0,
    step_duration_seconds=5
)

#si inizializza un episodio
obs, info = env_arrivi.reset(seed=42)

#si esegue uno step facendo muovere l'ascensore verso l'alto
#in questo modo evitiamo un eventuale imbarco durante il test
obs, reward, terminated, truncated, info = env_arrivi.step(1)

#si raccoglie il numero di passeggeri arrivati a ciascun piano
arrivi_per_piano = [
    len(env_arrivi.queues[i])
    for i in range(env_arrivi.n_floors)
]

#si stampa il numero di nuovi passeggeri arrivati a ogni piano
print("Arrivi per piano:", arrivi_per_piano)

#si stampa il numero totale di nuovi passeggeri arrivati
print("Arrivi totali:", sum(arrivi_per_piano))

#si verifica che il numero di passeggeri osservato corrisponda
#al numero di istanti di arrivo registrati in ogni coda
for floor in range(env_arrivi.n_floors):
    assert (
        obs[f"q{floor+1}"]
        == len(env_arrivi.queues[floor])
    )

#si verifica che tutti i nuovi arrivi siano stati registrati
#alla fine del primo step
for floor in range(env_arrivi.n_floors):
    assert all(
        arrival_step == 1
        for arrival_step in env_arrivi.queues[floor]
    )

#si verifica che l'attesa osservata sia nulla,
#in quanto tutti i passeggeri sono appena arrivati
for floor in range(env_arrivi.n_floors):
    assert obs[f"w{floor+1}"] == 0.0

#si verifica che l'osservazione sia ancora valida
assert env_arrivi.observation_space.contains(obs)

print("Test degli arrivi Poisson superato!")

Arrivi per piano: [7, 7, 7, 2, 6]
Arrivi totali: 29
Test degli arrivi Poisson superato!


#### Test sull'imbarco 

Viene adesso verificato che l'imbarco di un passeggero avvenga coerentemente a quanto definito in precedenza, ovvero: se l'ascensore è vuoto, ha scelto di restare fermo e la coda del piano non è vuota, allora l'ascensore deve prendere il passeggero. Per verificare questo e tutto ciò che esso comporta si eseguiranno i seguenti controlli:

- si verifica che il passeggero sia salito
- si verifica che la coda del piano sia vuota
- si verifica che la destinazione sia diversa da quella di arrivo
- si verifca che il tempo di attesa sia stato registrato
- si verifica che la reward sia nulla, dato che dopo l'imbarco non resta nessun passeggero in coda

In [75]:
#si crea un ambiente senza nuovi arrivi di passeggeri
env_test = gym.make(
    env_id,
    p_od=P_OD,
    arrival_rate_per_minute=0.0,
    max_steps=500
)

#si accede alla classe originale dell'ambiente
#per poter impostare manualmente lo stato interno
base_env = env_test.unwrapped

#si inizializza un nuovo episodio
obs, info = env_test.reset(seed=42)

#si posiziona l'ascensore al piano terra
base_env._agent_location = 0
base_env._target_location = 0
base_env._is_full = False

#si inizializzano tutte le code vuote
base_env.queues = [[] for _ in range(base_env.n_floors)]

#si inizializza lo storico dei tempi di attesa
base_env.waiting_times = []

#si inserisce un passeggero al piano terra
#che è arrivato all'inizio della simulazione
base_env.queues[0] = [0]

#si sceglie di restare fermi per verificare l'imbarco
obs, reward, terminated, truncated, info = env_test.step(0)

#si stampano i risultati del test
print("Osservazione:", obs)
print("Reward:", reward)
print(
    "Tempo di attesa registrato:",
    info["waiting_time_last_boarded"]
)

#si verifica che il passeggero sia salito
assert obs["passenger"] == 1
#si verifica che la coda del piano terra sia vuota
assert obs["q1"] == 0
#si verifica che la destinazione sia diversa dal piano di partenza
assert obs["target"] != 0
#si verifica che il tempo di attesa sia stato registrato
assert info["waiting_time_last_boarded"] == 5.0
#si verifica che la reward sia nulla, perché dopo l'imbarco
#non resta nessun passeggero in coda
assert reward == 0.0

print("Test di imbarco superato!")

env_test.close()

Osservazione: {'agent': np.int64(0), 'target': np.int64(3), 'passenger': np.int64(1), 'q1': np.int64(0), 'q2': np.int64(0), 'q3': np.int64(0), 'q4': np.int64(0), 'q5': np.int64(0), 'w1': np.float32(0.0), 'w2': np.float32(0.0), 'w3': np.float32(0.0), 'w4': np.float32(0.0), 'w5': np.float32(0.0)}
Reward: -0.0
Tempo di attesa registrato: 5.0
Test di imbarco superato!


#### Test sullo sbarco del passeggero

Viene verificato adesso lo sbarco, in particolare si verifica che:

- l'ascensore esegua correttamente le azioni per arrivare a destinazione
- il passeggero venga rilasciato
- la reward non venga più influenzata dal passeggero che sta viaggiando verso la destinazione

In [76]:
#si crea un ambiente senza nuovi arrivi di passeggeri
env_test = gym.make(
    env_id,
    p_od=P_OD,
    arrival_rate_per_minute=0.0,
    max_steps=500
)

#si accede all'ambiente originale per impostare manualmente lo stato
base_env = env_test.unwrapped

#si inizializza un nuovo episodio
obs, info = env_test.reset(seed=42)

#si posiziona l'ascensore al piano terra
base_env._agent_location = 0

#si imposta la destinazione al quarto piano
base_env._target_location = 3

#si imposta la presenza di un passeggero a bordo
base_env._is_full = True

#si inizializzano tutte le code vuote
base_env.queues = [[] for _ in range(base_env.n_floors)]

#si inizializza lo storico dei tempi di attesa
base_env.waiting_times = []

#si definisce una sequenza di tre azioni di salita
#per raggiungere il piano di destinazione
actions = [1, 1, 1]

#si esegue il test dello sbarco
for action in actions:

    #si salva il piano precedente
    old_floor = base_env._agent_location

    #si esegue uno step della simulazione
    obs, reward, terminated, truncated, info = env_test.step(action)

    print(
        f"Azione: {action} | "
        f"Piano: {old_floor} -> {obs['agent']} | "
        f"Target: {obs['target']} | "
        f"Passeggero: {obs['passenger']} | "
        f"Reward: {reward:.2f}"
    )

#si verifica che l'ascensore abbia raggiunto il target
assert obs["agent"] == 3
#si verifica che il passeggero sia sceso
assert obs["passenger"] == 0
#si verifica che, a ascensore vuoto, il target coincida con la posizione
assert obs["target"] == obs["agent"]

print("Test di sbarco superato!")

env_test.close()

Azione: 1 | Piano: 0 -> 1 | Target: 3 | Passeggero: 1 | Reward: -0.00
Azione: 1 | Piano: 1 -> 2 | Target: 3 | Passeggero: 1 | Reward: -0.00
Azione: 1 | Piano: 2 -> 3 | Target: 3 | Passeggero: 0 | Reward: -0.00
Test di sbarco superato!


#### Test Sbarco e Imbarco

Si verifica che:

- il passeggero a bordo sbarchi al piano di destinazione
- il passeggero in coda sale automaticamente

In [77]:
#si disabilitano gli arrivi casuali
env_debug.arrival_rate = np.zeros(env_debug.n_floors)

#si inizializza l'ambiente
obs, info = env_debug.reset(seed=42)

#si posiziona l'ascensore al secondo piano con un passeggero diretto al terzo
env_debug._agent_location = 1
env_debug._target_location = 2
env_debug._is_full = True

#si inserisce un passeggero in attesa al terzo piano
env_debug.queues[2] = [env_debug.current_step]

#si raggiunge il piano di destinazione
obs, reward, terminated, truncated, info = env_debug.step(1)

#si verifica lo sbarco e l'imbarco automatico
assert env_debug._agent_location == 2
assert env_debug._is_full is True
assert len(env_debug.queues[2]) == 0

print("Test superato: il passeggero in coda è salito automaticamente.")

Test superato: il passeggero in coda è salito automaticamente.


#### Test sulla Reward 

Si verifica che la reward venga calcolata correttamente, per farlo:

- si simula l'arrivo di due passeggeri su piani e a tempi diversi
- si imposta lo step a un certo istante x
- si impone che l'ascensore allo step x+1 non resti fermo (impedendo l'imbarco di passeggeri)
- si raccoglie la reward

La reward viene calcolata sullo stato al termine dello step: in questo test nessun passeggero sale e non ci sono nuovi arrivi, perciò coincide con l'attesa accumulata dai due passeggeri in coda.


In [78]:
#si crea un ambiente senza arrivi casuali
env_reward = ElevatorEnv(
    p_od=P_OD,
    arrival_rate_per_minute=0.0,
    step_duration_seconds=5,
    age_weight=0.5,
    reference_wait_seconds=60.0
)

#si inizializza l'ambiente
obs, info = env_reward.reset(seed=42)

#si inseriscono tre passeggeri arrivati agli step 0, 2 e 4
env_reward.queues = [[0, 2, 4], [], [], [], []]

#si imposta lo step corrente a 9
env_reward.current_step = 9

#si esegue uno step senza consentire l'imbarco
obs, reward, terminated, truncated, info = env_reward.step(1)

#al termine dello step, i tempi di attesa sono 50, 40 e 30 secondi
#la loro somma è quindi pari a 120 secondi
expected_waiting_time = 120.0

#si calcola la reward attesa:
#-(3 + 0.5 * 120 / 60) = -4
expected_reward = -4.0

#si verificano il numero di passeggeri e il tempo totale di attesa
assert info["queues_total"] == 3
assert np.isclose(info["total_waiting_age"], expected_waiting_time)

#si verifica la reward
assert np.isclose(reward, expected_reward)

print("Passeggeri in coda:", info["queues_total"])
print("Tempo totale di attesa:", info["total_waiting_age"], "secondi")
print("Reward ottenuta:", reward)
print("Test della reward superato!")

env_reward.close()

Passeggeri in coda: 3
Tempo totale di attesa: 120.0 secondi
Reward ottenuta: -4.0
Test della reward superato!


#### Test sull'osservazione

Si verifica che l'osservazione riporti correttamente, per ogni piano, il numero di passeggeri in coda e l'attesa del passeggero più vecchio. In particolare si verifica che:

- il conteggio e l'attesa osservati corrispondano allo stato interno delle code
- una coda vuota abbia attesa nulla
- dopo un imbarco il passeggero che sale sia il più vecchio e l'attesa osservata si riferisca al nuovo passeggero in testa alla coda


In [79]:
#si crea un ambiente senza nuovi arrivi casuali
env_obs = ElevatorEnv(
    p_od=P_OD,
    arrival_rate_per_minute=0.0,
    step_duration_seconds=5
)

obs, info = env_obs.reset(seed=42)

#si impostano tre passeggeri in coda al terzo piano,
#arrivati agli step 0, 2 e 3, e uno al quinto piano,
#arrivato allo step 3
env_obs.queues = [[], [], [0, 2, 3], [], [3]]

#si imposta il contatore al quarto step
env_obs.current_step = 4

#si posiziona l'ascensore vuoto al terzo piano
env_obs._agent_location = 2
env_obs._target_location = 2
env_obs._is_full = False

#si verifica l'osservazione prima dell'imbarco:
#il più vecchio al terzo piano attende da 4 step, cioè 20 secondi,
#quello del quinto piano da 1 step, cioè 5 secondi
obs = env_obs._get_obs()

assert obs["q3"] == 3 and obs["w3"] == 20.0
assert obs["q5"] == 1 and obs["w5"] == 5.0

#le code vuote hanno attesa nulla
assert obs["q1"] == 0 and obs["w1"] == 0.0
assert obs["q2"] == 0 and obs["w2"] == 0.0
assert obs["q4"] == 0 and obs["w4"] == 0.0

#si sceglie di restare fermi, così da imbarcare il passeggero più vecchio
obs, reward, terminated, truncated, info = env_obs.step(0)

#il passeggero imbarcato era in coda dallo step 0:
#al quinto step ha atteso 25 secondi
assert info["waiting_time_last_boarded"] == 25.0

#rimangono i passeggeri arrivati agli step 2 e 3,
#il più vecchio dei quali attende ora da 3 step, cioè 15 secondi
assert obs["q3"] == 2 and obs["w3"] == 15.0

#l'attesa del passeggero al quinto piano è aumentata di 5 secondi
assert obs["q5"] == 1 and obs["w5"] == 10.0

print("Test dell'osservazione superato!")

env_obs.close()

Test dell'osservazione superato!


#### Test di coerenza dello stato

Si verifica adesso che lo stato dell'ambiente resti coerente dopo un numero X di steps. In particolare, a ogni step, si verifica che:

- l'osservazione sia coerente con le code interne (numero di passeggeri e attesa del più vecchio)
- gli istanti di arrivo di ogni coda siano ordinati e non successivi allo step corrente
- la posizione dell'ascensore sia valida
- la reward sia un numero finito
- l'osservazione sia compatibile con lo spazio dichiarato

In [80]:
#si crea un ambiente per verificare la coerenza dello stato
env_debug = ElevatorEnv(
    p_od=P_OD,
    arrival_rate_per_minute=0.5,
    step_duration_seconds=5,
    max_steps=500
)

#si inizializza un episodio con un seed riproducibile
obs, info = env_debug.reset(seed=123)

#si crea un generatore casuale per scegliere le azioni
rng = np.random.default_rng(123)

#si eseguono 200 step per verificare il comportamento dell'ambiente
for test_step in range(200):

    #si sceglie casualmente una delle tre azioni possibili
    action = int(rng.integers(0, 3))

    #si esegue uno step della simulazione
    obs, reward, terminated, truncated, info = env_debug.step(action)

    #si verifica che l'osservazione sia coerente con le code interne:
    #il numero di passeggeri in coda e l'attesa del più vecchio
    for floor in range(env_debug.n_floors):

        queue = env_debug.queues[floor]

        assert (
            obs[f"q{floor+1}"] == len(queue)
        ), f"Incoerenza nella coda del piano {floor}"

        expected_wait = (
            (env_debug.current_step - queue[0])
            * env_debug.step_duration_seconds
            if queue else 0.0
        )

        assert (
            obs[f"w{floor+1}"] == expected_wait
        ), f"Incoerenza nell'attesa del piano {floor}"

    #si verifica che gli istanti di arrivo di ogni coda siano ordinati
    #e non successivi allo step corrente, così il primo
    #passeggero della coda è sempre quello più vecchio
    for queue in env_debug.queues:
        assert queue == sorted(queue)
        assert all(
            arrival_step <= env_debug.current_step
            for arrival_step in queue
        )

    #si verifica che la posizione dell'ascensore sia valida
    assert 0 <= env_debug._agent_location < env_debug.n_floors

    #si verifica che il target sia un piano valido
    assert 0 <= env_debug._target_location < env_debug.n_floors

    #si verifica che la reward sia un numero finito
    assert np.isfinite(reward)

    #si verifica che l'osservazione sia compatibile con lo spazio dichiarato
    assert env_debug.observation_space.contains(obs)

    #si interrompe il test se l'episodio è terminato
    if terminated or truncated:
        break


print("Test di coerenza dello stato superato!")

#si stampa il numero di step effettivamente eseguiti
print("Step eseguiti:", env_debug.current_step)
#si stampa il numero totale di passeggeri ancora in coda
print(
    "Passeggeri in coda:",
    sum(len(queue) for queue in env_debug.queues)
)

env_debug.close()

Test di coerenza dello stato superato!
Step eseguiti: 200
Passeggeri in coda: 35
